# Smart MCQ Solver Challenge, final notebook

Course: Deep Learning and Generative AI Project, Diploma Level  
Roll number: 22f3000114  
Competition: Smart MCQ Solver Challenge  
Metric: MAP@3

This notebook contains the full pipeline end to end: data checks, cleaning, and three models of
different architectures. It runs in two modes.

**Build mode.** Set `BUILD = True`. Trains the from scratch model, runs retrieval, computes
probabilities for every Llama signal, and saves all of it to `/kaggle/working`. Run this once.

**Submit mode.** Set `BUILD = False`. Loads the saved arrays and writes `submission.csv` for
whichever model is selected in `SUBMIT_MODEL`. Runs in under a minute and needs no GPU.
Three submissions are made by changing `SUBMIT_MODEL` and rerunning.

## The three models

| Model | Type | Section |
|---|---|---|
| BiLSTM classifier | Built from scratch, no pretrained weights | 5 |
| Llama 3.1 8B with retrieval | Pretrained, 4 bit quantised, RAG context | 6 |
| Four signal ensemble | Base and QLoRA tuned, with and without context | 7 |

## Inputs to attach

1. Competition data
2. `jjinho/wikipedia-2023-07-faiss-index`
3. `jjinho/wikipedia-20230701`
4. Models tab: `metaresearch/llama-3.1`, Transformers, 8b-instruct, version 2
5. Models tab: `srg9000/all-minilm-l6-v2`

Accelerator: GPU T4 x2. The `WANDB` secret is needed for experiment tracking.

Models load from attached Kaggle paths rather than the Hugging Face hub. Hub downloads of an 8B
model restart on every session and cost most of the session budget, so the attached copies are
used instead. This is an engineering constraint, not a modelling choice.

## 1. Setup

Packages are installed only when missing. The Kaggle image already ships most of them, and a
plain `pip install` on every run wastes minutes of the session.

In [1]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Install pkg with pip only if it cannot already be imported."""
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for pkg, mod in [('bitsandbytes', 'bitsandbytes'), ('peft', 'peft'), ('kagglehub', 'kagglehub'),
                 ('sentence-transformers', 'sentence_transformers'), ('faiss-cpu', 'faiss'),
                 ('wandb', 'wandb')]:
    ensure(pkg, mod)
print('Package check done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.7 MB/s eta 0:00:00
Package check done.


In [2]:
import os, gc, re, glob, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

OPTIONS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP = {o: i for i, o in enumerate(OPTIONS)}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '| GPUs:', torch.cuda.device_count())

Device: cuda | GPUs: 2


Two switches control the whole notebook. `BUILD` decides whether the expensive cells run or the
saved artifacts are loaded. `SUBMIT_MODEL` picks which model writes `submission.csv`.

In [3]:
BUILD = True          # True trains and saves everything, False loads saved arrays

SUBMIT_MODEL = 'ensemble'   # one of: 'bilstm', 'base_rag', 'ensemble'

WORK = '/kaggle/working'

# retrieval settings
K_ARTICLES = 10       # wikipedia articles fetched per option query
K_FINAL    = 5        # chunks kept as context for one question
CTX_CHARS  = 4500     # character cap on the context block

# from scratch model settings
MAX_LEN    = 512
EMBED_DIM  = 128
HIDDEN_DIM = 256
N_EPOCHS   = 20
BATCH_SIZE = 32
LR         = 1e-3

print('BUILD =', BUILD, '| SUBMIT_MODEL =', SUBMIT_MODEL)

BUILD = True | SUBMIT_MODEL = ensemble


Paths are resolved by searching `/kaggle/input` rather than being hardcoded. Kaggle changes the
input folder name between sessions, and a hardcoded path breaks silently when that happens.

In [4]:
def find_file(name):
    """Return the first path under /kaggle/input matching a filename."""
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found, check the attached inputs')
    return hits[0]

def find_model_dir(keyword):
    """Return the folder of the first config.json under /kaggle/input matching a keyword."""
    for h in glob.glob('/kaggle/input/**/config.json', recursive=True):
        if keyword in h.lower():
            return os.path.dirname(h)
    raise FileNotFoundError(f'{keyword} model not found, attach it from the Models tab')

## 2. Load the competition data

In [5]:
train = pd.read_csv(find_file('train.csv'))
test  = pd.read_csv(find_file('test.csv'))

train['label'] = train['answer'].map(LABEL_MAP)
assert train['label'].notna().all(), 'an answer value is outside A to E'

print('train:', train.shape, '| test:', test.shape)
train.head(3)

train: (2000, 9) | test: (500, 7)


,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2


## 3. Exploratory data analysis

### 3.1 Shape, labels and text lengths

The first thing to check is whether any answer letter dominates. If one letter is much more
common than the rest, a model that always predicts that letter becomes a baseline worth beating,
and MAP@3 numbers have to be read against it.

In [6]:
print('rows:', len(train), '| columns:', list(train.columns))
print()

counts = train['answer'].value_counts().sort_index()
share = (counts / len(train) * 100).round(2)
print('answer distribution')
print(pd.DataFrame({'count': counts, 'percent': share}))

majority = share.max() / 100
print()
print(f'majority class baseline accuracy: {majority:.4f}')
print(f'majority class baseline MAP@3   : {majority + (1/5)*0.5 + (1/5)*(1/3):.4f} approx')

rows: 2000 | columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'label']

answer distribution
        count  percent
answer                
A         369    18.45
B         490    24.50
C         459    22.95
D         358    17.90
E         324    16.20

majority class baseline accuracy: 0.2450
majority class baseline MAP@3   : 0.4117 approx


In [7]:
lengths = pd.DataFrame({
    'prompt_words':  train['prompt'].str.split().str.len(),
    'option_words':  train[OPTIONS].apply(lambda r: np.mean([len(str(v).split()) for v in r]), axis=1),
})
lengths['total_words'] = lengths['prompt_words'] + 5 * lengths['option_words']

print(lengths.describe().round(1))
print()
for q in [0.5, 0.9, 0.95, 0.99]:
    print(f'p{int(q*100)} total words: {lengths["total_words"].quantile(q):.0f}')

       prompt_words  option_words  total_words
count        2000.0        2000.0       2000.0
mean           18.1          26.3        149.5
std             6.8          17.0         85.4
min             3.0           1.0         15.0
25%            14.0          14.6         88.0
50%            17.0          23.0        133.0
75%            22.0          36.8        202.0
max            51.0          85.4        457.0

p50 total words: 133
p90 total words: 257
p95 total words: 322
p99 total words: 412


Half the rows sit near 130 words but the tail runs past 400. Subword tokenisation produces
roughly 1.4 tokens per word, so `MAX_LEN` has to cover the tail rather than the median. At 256
tokens about a third of the rows would be cut, and because the options are appended after the
prompt, the text lost is options D and E. A model cannot rank an option it never read. `MAX_LEN`
is set to 512, which leaves around 32 rows truncated.

### 3.2 Duplicate rows

The row count is 2000, but that does not mean 2000 distinct questions. Checking for exact
duplicates across the prompt and all five options is the first test.

In [8]:
dupe_mask = train.duplicated(subset=['prompt'] + OPTIONS, keep=False)
print('rows that are part of an exact duplicate group:', int(dupe_mask.sum()))
print('unique prompts:', train['prompt'].nunique(), 'out of', len(train), 'rows')

rows that are part of an exact duplicate group: 344
unique prompts: 1758 out of 2000 rows


### 3.3 Where the duplicates come from

Looking at a duplicated group shows the pattern. The same question and the same five options
appear more than once, and the only difference is the instruction phrase at the front of the
prompt.

In [9]:
example = train[dupe_mask].sort_values('prompt').head(2)
for _, r in example.iterrows():
    print(repr(r['prompt'][:110]))
    print('   answer:', r['answer'], '| option A:', str(r['A'])[:70])
    print()

'Choose the correct answer: What are permutation-inversion groups?'
   answer: E | option A: Permutation-inversion groups are groups of symmetry operations that ar

'Choose the correct answer: What are permutation-inversion groups?'
   answer: E | option A: Permutation-inversion groups are groups of symmetry operations that ar



The prompts differ only in their opening phrase. Stripping that phrase and counting again shows
how many questions are actually distinct.

In [10]:
def core_question(prompt):
    """Strip a leading instruction phrase such as 'Select the most accurate option:'."""
    return re.sub(r'^[^:]{0,60}:\s*', '', str(prompt)).strip()

train['core_q'] = train['prompt'].apply(core_question)

print('rows                 :', len(train))
print('unique prompts       :', train['prompt'].nunique())
print('unique core questions:', train['core_q'].nunique())
print()
print('template prefixes in use:')
prefixes = train['prompt'].str.extract(r'^([^:]{0,60}:)')[0].value_counts()
print(prefixes.head(10))

rows                 : 2000
unique prompts       : 1758
unique core questions: 1125

template prefixes in use:
0
Pick the best possible answer:      316
Select the most accurate option:    309
Identify the correct statement:     302
Determine the correct option:       299
Choose the correct answer:          292
Name: count, dtype: int64


The dataset was built by wrapping a smaller pool of questions in several instruction phrases. The
row count overstates how much distinct material is available.

This matters for validation. A split that treats rows as independent puts one phrasing of a
question in the training set and another phrasing of the same question in the validation set. The
model then scores well on validation because it has already seen that question, not because it
generalises. The fix is to group by `core_q` so that every phrasing of a question stays on one
side of the split.

### 3.4 Cleaning

Exact duplicates are dropped since they carry no extra information. Rows that share a core
question but differ in their options are kept, because the option sets genuinely differ. Those
rows are held together by the split instead.

In [11]:
before = len(train)
train = train.drop_duplicates(subset=['prompt'] + OPTIONS, keep='first').reset_index(drop=True)
print(f'dropped {before - len(train)} exact duplicate rows, {len(train)} remain')
print('unique core questions after cleaning:', train['core_q'].nunique())

dropped 183 exact duplicate rows, 1817 remain
unique core questions after cleaning: 1125


## 4. Train and validation split

`GroupShuffleSplit` keeps every row sharing a core question on the same side. The assertion is
the point of the cell: it fails loudly if any question leaks across, rather than letting an
inflated score go unnoticed.

In [12]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, va_idx = next(gss.split(train, groups=train['core_q']))

train_df = train.iloc[tr_idx].reset_index(drop=True)
val_df   = train.iloc[va_idx].reset_index(drop=True)

overlap = set(train_df['core_q']) & set(val_df['core_q'])
assert len(overlap) == 0, f'{len(overlap)} core questions leak across the split'

print('train:', len(train_df), '| val:', len(val_df))
print('core question overlap:', len(overlap))
print()
print('validation label balance')
print((val_df['answer'].value_counts(normalize=True).sort_index() * 100).round(1))

train: 1473 | val: 344
core question overlap: 0

validation label balance
answer
A    21.2
B    26.2
C    21.5
D    13.4
E    17.7
Name: proportion, dtype: float64


The split is not exactly 80 to 20 because groups differ in size. Correct grouping matters more
than a round ratio.

`GroupShuffleSplit` does not stratify by label, so the balance is printed as a check. Every
letter sitting in a reasonable band means the validation set is usable as it stands.

## 5. The evaluation metric

MAP@3 gives full credit when the correct answer is ranked first, half when it is second, and a
third when it is third. Nothing below that scores. Every model in this notebook is measured with
the same function so the numbers are comparable.

In [13]:
def map_at_3(predictions, labels):
    """Mean average precision at 3, predictions are ranked lists of option letters."""
    weights = [1.0, 1/2, 1/3]
    total = 0.0
    for preds, true in zip(predictions, labels):
        for i, p in enumerate(preds[:3]):
            if p == true:
                total += weights[i]
                break
    return total / len(labels)

def top3(probs):
    """Return the three highest scoring option letters, best first."""
    return [OPTIONS[i] for i in np.argsort(-np.asarray(probs))[:3]]

assert map_at_3([['A','B','C']], ['A']) == 1.0
assert map_at_3([['B','A','C']], ['A']) == 0.5
assert abs(map_at_3([['B','C','A']], ['A']) - 1/3) < 1e-9
assert map_at_3([['B','C','D']], ['A']) == 0.0
print('map_at_3 passes all four ranking cases')

map_at_3 passes all four ranking cases


## 6. Model 1: BiLSTM built from scratch

This model is written from scratch. Every weight starts random and is trained here. Nothing
pretrained is loaded.

The architecture is an embedding layer, a two layer bidirectional LSTM, masked mean pooling over
the timesteps, and a small classifier head that outputs five scores. The question and its five
options are joined into one string, so the task becomes five way classification and the loss is
plain cross entropy.

A bidirectional LSTM is used rather than a plain RNN because the gates carry information across
longer spans without the gradient vanishing, which matters when the answer depends on a phrase
early in a long option list.

### 6.1 Tokenisation

The DeBERTa tokenizer is reused to turn text into ids. Only the tokenizer is reused. No DeBERTa
weights are loaded, and the embedding table in this model is initialised randomly and trained
from scratch. Subword tokenisation handles unseen scientific vocabulary better than a word level
vocabulary built from 1400 rows.

In [14]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')
assert tok.pad_token_id == 0, f'pad id is {tok.pad_token_id}, update the mask in the model'

def format_mcq(row):
    """Return the question text and the five options joined into one string."""
    prompt = str(row['prompt'])
    options = ' '.join(f'{o}) {row[o]}' for o in OPTIONS)
    return prompt, options

def encode_frame(df, with_labels=True):
    """Tokenise a dataframe into padded id and mask tensors, plus labels when present."""
    pairs = [format_mcq(r) for _, r in df.iterrows()]
    prompts, options = zip(*pairs)
    enc = tok(list(prompts), list(options), padding='max_length',
              truncation=True, max_length=MAX_LEN, return_tensors='pt')
    out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}
    if with_labels:
        out['labels'] = torch.tensor(df['label'].values, dtype=torch.long)
    return out

print('vocabulary size:', len(tok))

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

vocabulary size: 128001


In [15]:
class MCQDataset(torch.utils.data.Dataset):
    """Wraps encoded tensors so a DataLoader can batch them."""
    def __init__(self, enc):
        self.enc = enc
    def __len__(self):
        return self.enc['input_ids'].size(0)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}

if BUILD:
    train_loader = DataLoader(MCQDataset(encode_frame(train_df)), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(MCQDataset(encode_frame(val_df)),   batch_size=BATCH_SIZE, shuffle=False)
    print('batches, train:', len(train_loader), '| val:', len(val_loader))

batches, train: 47 | val: 11


### 6.2 Architecture

Real sequence lengths come from the attention mask and are passed to `pack_padded_sequence`, so
the LSTM never reads padding. Pooling is a masked mean over valid timesteps, which keeps padded
positions out of the average.

In [16]:
class BiLSTMClassifier(nn.Module):
    """Embedding, bidirectional LSTM, masked mean pooling, five way classifier head."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=5,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes))

    def forward(self, input_ids, lengths):
        """Return one logit per option, shape batch by five."""
        embed = self.embed_dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(embed, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        rnn_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=input_ids.size(1))
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (rnn_out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.classifier(pooled)

# shape check before any training
_m = BiLSTMClassifier(vocab_size=1000, embed_dim=32, hidden_dim=64)
_out = _m(torch.randint(1, 1000, (4, 20)), torch.tensor([20, 15, 10, 5]))
assert _out.shape == (4, 5), f'expected (4, 5), got {tuple(_out.shape)}'
print('forward pass shape check passed:', tuple(_out.shape))
del _m, _out

forward pass shape check passed: (4, 5)


### 6.3 Training

A plain PyTorch loop, one function to train an epoch and one to evaluate. Accuracy, macro F1 and
MAP@3 are computed every epoch and logged to Weights and Biases. The checkpoint is written only
when validation MAP@3 improves.

In [17]:
def train_one_epoch(model, loader, optimizer, device):
    """Run one training epoch and return the mean loss."""
    model.train()
    total = 0.0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['attention_mask'].sum(dim=1)

        optimizer.zero_grad()
        logits = model(input_ids, lengths)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    """Return accuracy, macro F1, MAP@3 and the probability array for a loader."""
    model.eval()
    preds, labels, probs = [], [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        lengths = batch['attention_mask'].sum(dim=1)
        p = F.softmax(model(input_ids, lengths), dim=1).cpu().numpy()
        probs.extend(p)
        preds.extend(p.argmax(axis=1))
        labels.extend(batch['labels'].numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    m3 = map_at_3([top3(p) for p in probs], [OPTIONS[i] for i in labels])
    return acc, f1, m3, np.array(probs)

In [18]:
if BUILD:
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret('WANDB'))
    except Exception as e:
        print('W&B login skipped:', e)

    run = wandb.init(entity='22f3000114-dl-genai-project', project='dl-genai-project',
                     name='bilstm-scratch-grouped', group='final-notebook',
                     config={'model': 'biLSTM from scratch', 'embed_dim': EMBED_DIM,
                             'hidden_dim': HIDDEN_DIM, 'max_len': MAX_LEN, 'lr': LR,
                             'split': 'GroupShuffleSplit on core_q'}, reinit=True)

    bilstm = BiLSTMClassifier(vocab_size=len(tok), embed_dim=EMBED_DIM,
                              hidden_dim=HIDDEN_DIM, pad_idx=tok.pad_token_id).to(device)
    n_params = sum(p.numel() for p in bilstm.parameters() if p.requires_grad)
    print('trainable parameters:', f'{n_params:,}')

    optimizer = torch.optim.AdamW(bilstm.parameters(), lr=LR)
    best_map3 = -1.0

    for epoch in range(N_EPOCHS):
        loss = train_one_epoch(bilstm, train_loader, optimizer, device)
        tr_acc, tr_f1, tr_map3, _ = evaluate(bilstm, train_loader, device)
        acc, f1, m3, _ = evaluate(bilstm, val_loader, device)

        run.log({'epoch': epoch, 'train_loss': loss, 'train_accuracy': tr_acc,
                 'val_accuracy': acc, 'val_f1_macro': f1, 'val_map3': m3})
        print(f'epoch {epoch:2d}  loss {loss:.4f}  train_acc {tr_acc:.4f}  '
              f'val_acc {acc:.4f}  val_f1 {f1:.4f}  val_map3 {m3:.4f}')

        if m3 > best_map3:
            best_map3 = m3
            torch.save(bilstm.state_dict(), f'{WORK}/bilstm_best.pt')

    run.log({'best_val_map3': best_map3})
    run.finish()
    print('best validation MAP@3:', round(best_map3, 4))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260721_151746-8spbnfid
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm-scratch-grouped
wandb: ⭐️ View project at https://wandb.ai/22f3000114-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/22f3000114-dl-genai-project/dl-genai-pr

trainable parameters: 18,884,229
epoch  0  loss 1.4162  train_acc 0.4902  val_acc 0.3895  val_f1 0.3186  val_map3 0.5538
epoch  1  loss 0.8452  train_acc 0.9199  val_acc 0.7733  val_f1 0.7610  val_map3 0.8435
epoch  2  loss 0.2485  train_acc 0.9932  val_acc 0.9709  val_f1 0.9735  val_map3 0.9753
epoch  3  loss 0.1012  train_acc 0.9518  val_acc 0.9215  val_f1 0.9130  val_map3 0.9569
epoch  4  loss 0.1040  train_acc 0.9986  val_acc 0.9797  val_f1 0.9811  val_map3 0.9864
epoch  5  loss 0.0222  train_acc 0.9986  val_acc 0.9767  val_f1 0.9803  val_map3 0.9884
epoch  6  loss 0.0179  train_acc 0.9993  val_acc 0.9797  val_f1 0.9811  val_map3 0.9898
epoch  7  loss 0.0046  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000
epoch  8  loss 0.0044  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000
epoch  9  loss 0.0029  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000
epoch 10  loss 0.0049  train_acc 0.9993  val_acc 0.9797  val_f1 0.9811  val_map3 0.9898

wandb: updating run metadata


epoch 19  loss 0.0069  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000


wandb: uploading history steps 18-20, summary, console lines 19-20
wandb: 
wandb: Run history:
wandb:  best_val_map3 ▁
wandb:          epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb: train_accuracy ▁▇█▇████████████████
wandb:     train_loss █▅▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val_accuracy ▁▅█▇████████████████
wandb:   val_f1_macro ▁▆█▇████████████████
wandb:       val_map3 ▁▆█▇████████████████
wandb: 
wandb: Run summary:
wandb:  best_val_map3 1
wandb:          epoch 19
wandb: train_accuracy 1
wandb:     train_loss 0.00687
wandb:   val_accuracy 1
wandb:   val_f1_macro 1
wandb:       val_map3 1
wandb: 
wandb: 🚀 View run bilstm-scratch-grouped at: https://wandb.ai/22f3000114-dl-genai-project/dl-genai-project/runs/8spbnfid
wandb: ⭐️ View project at: https://wandb.ai/22f3000114-dl-genai-project/dl-genai-project
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260721_151746-8spbnfid/logs


best validation MAP@3: 1.0


Train accuracy is logged next to validation accuracy on purpose. A high training score with a
lower validation score is ordinary overfitting on a small dataset. The two moving together and
reaching a perfect score is a different problem and means information is leaking into validation.

An earlier version of this model reached a perfect validation score. The cause was the split, not
the architecture. Splitting on raw prompts left the same question, wrapped in a different
instruction phrase, on both sides. Grouping on `core_q` in section 4 is what fixes it, and it is
the reason the assertion in that cell exists.

### 6.4 Test predictions from the from scratch model

In [19]:
if BUILD:
    bilstm.load_state_dict(torch.load(f'{WORK}/bilstm_best.pt'))
    bilstm.eval()

    test_enc = encode_frame(test, with_labels=False)
    test_probs_bilstm = []
    with torch.no_grad():
        for i in range(0, len(test), BATCH_SIZE):
            ids = test_enc['input_ids'][i:i+BATCH_SIZE].to(device)
            lens = test_enc['attention_mask'][i:i+BATCH_SIZE].sum(dim=1)
            p = F.softmax(bilstm(ids, lens), dim=1).cpu().numpy()
            test_probs_bilstm.extend(p)

    test_probs_bilstm = np.array(test_probs_bilstm)
    np.save(f'{WORK}/test_probs_bilstm.npy', test_probs_bilstm)
    print('saved test_probs_bilstm.npy', test_probs_bilstm.shape)

    del bilstm, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

saved test_probs_bilstm.npy (500, 5)


## 7. The Wikipedia corpus and retrieval

The questions are science questions whose answers are facts, not things that can be worked out
from the wording. The similarity baselines from the earlier milestones underperformed because the
distractors are written to share vocabulary with the question, so word overlap does not separate
them. Supplying the model with the relevant passage is what changes that.

The corpus is a July 2023 Wikipedia dump with a prebuilt FAISS index over MiniLM embeddings. The
index holds one vector per article, and the article text sits in parquet files alongside it.

In [20]:
if BUILD:
    import faiss
    from sentence_transformers import SentenceTransformer

    WIKI_INDEX_PATH = find_file('wikipedia_202307.index')
    wiki_map_path = find_file('wiki_2023_index.parquet')
    WIKI_PARQUET_DIR = os.path.dirname(wiki_map_path)

    print('loading the FAISS index, this takes about a minute')
    wiki_index = faiss.read_index(WIKI_INDEX_PATH)
    wiki_map = pd.read_parquet(wiki_map_path)

    print('indexed article vectors:', f'{wiki_index.ntotal:,}')
    print('embedding dimension     :', wiki_index.d)
    print('map columns             :', list(wiki_map.columns))
    print('parquet shards          :', len(glob.glob(os.path.join(WIKI_PARQUET_DIR, '*.parquet'))))
    wiki_map.head(3)

loading the FAISS index, this takes about a minute
indexed article vectors: 6,286,775
embedding dimension     : 384
map columns             : ['id', 'title', 'text', 'categories', 'context', 'file']
parquet shards          : 29


The index covers millions of articles, far more than the questions need. Retrieval is what
narrows it down. Each option is combined with the question to form its own query, so an option
mentioning a specific term can pull in the article about that term even when the question does
not name it. Five queries per question, ten articles each, then the pool is ranked and trimmed.

### 7.1 Stage 1, find candidate articles

The bi encoder embeds every query and FAISS returns the nearest article vectors. This is the
coarse stage, ranking millions of articles down to fifty candidates per question.

In [21]:
if BUILD:
    MINILM_DIR = find_model_dir('minilm')
    bi_encoder = SentenceTransformer(MINILM_DIR, device=device)
    print('bi encoder loaded from', MINILM_DIR)

    def build_queries(row):
        """Return one query string per option, each combining the prompt with that option."""
        p = str(row['prompt'])
        return [f'{p} {row[o]}' for o in OPTIONS]

    val_rows  = [val_df.iloc[i] for i in range(len(val_df))]
    test_rows = [test.iloc[i] for i in range(len(test))]
    all_rows  = val_rows + test_rows
    N_VAL = len(val_rows)

    queries = []
    for r in all_rows:
        queries.extend(build_queries(r))

    q_emb = bi_encoder.encode(queries, batch_size=64, show_progress_bar=True)
    q_emb = np.asarray(q_emb, dtype='float32')
    _, art = wiki_index.search(q_emb, K_ARTICLES)
    article_rows = art.reshape(len(all_rows), 5 * K_ARTICLES)

    print('questions:', len(all_rows), '| candidate articles each:', article_rows.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bi encoder loaded from /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2


Batches:   0%|          | 0/66 [00:00<?, ?it/s]

questions: 844 | candidate articles each: 50


### 7.2 Stage 2, load the article text

Only the articles that were actually retrieved are read out of the parquet shards. Results are
cached to disk by article id so a rerun skips the work. The coverage assertion catches the case
where ids fail to match between the index and the parquet files, which silently produces empty
contexts if it goes unchecked.

In [22]:
if BUILD:
    import pyarrow.parquet as pq

    def save_ckpt(name, obj):
        """Pickle an object into the working directory."""
        with open(f'{WORK}/{name}.pkl', 'wb') as f:
            pickle.dump(obj, f)

    def load_ckpt(name):
        """Load a pickle from the working directory, or None when it is absent."""
        p = f'{WORK}/{name}.pkl'
        return pickle.load(open(p, 'rb')) if os.path.exists(p) else None

    def chunk_article(title, text):
        """Split an article into paragraph chunks, each prefixed with the title."""
        parts = [p.strip() for p in str(text).split('\n') if len(p.strip()) > 80]
        if not parts:
            t = str(text)
            parts = [t[i:i+800] for i in range(0, min(len(t), 30000), 700)]
        return [f'{title}. {p}'[:1500] for p in parts]

    needed = wiki_map.iloc[article_rows.flatten()].copy()
    file_col = 'file' if 'file' in needed.columns else needed.columns[-1]
    id_col   = 'id'   if 'id'   in needed.columns else needed.columns[0]
    needed_by_file = needed.groupby(file_col)[id_col].apply(
        lambda s: set(str(x) for x in s)).to_dict()

    article_chunks = load_ckpt('article_chunks') or {}
    for fname, id_set in tqdm(needed_by_file.items(), desc='parquet shards'):
        missing = {i for i in id_set if i not in article_chunks}
        if not missing:
            continue
        fpath = os.path.join(WIKI_PARQUET_DIR, str(fname))
        if not os.path.exists(fpath):
            hits = glob.glob(os.path.join(WIKI_PARQUET_DIR, '**', str(fname)), recursive=True)
            if not hits:
                continue
            fpath = hits[0]
        pf = pq.ParquetFile(fpath)
        tcol = 'text' if 'text' in pf.schema.names else [
            c for c in pf.schema.names if c not in ('id', 'title')][-1]
        cols = [c for c in ['id', 'title', tcol] if c in pf.schema.names]
        for batch in pf.iter_batches(batch_size=5000, columns=cols):
            bdf = batch.to_pandas()
            bdf['id'] = bdf['id'].astype(str)
            hit = bdf[bdf['id'].isin(missing)]
            for _, r in hit.iterrows():
                title = str(r['title']) if 'title' in bdf.columns else ''
                article_chunks[r['id']] = chunk_article(title, r[tcol])
            del bdf, hit
        del pf
        gc.collect()
        save_ckpt('article_chunks', article_chunks)

    wanted = set(str(x) for x in needed[id_col])
    cov = len(wanted & set(article_chunks))
    print(f'article coverage: {cov} of {len(wanted)} ({100*cov/len(wanted):.1f} percent)')
    assert cov / len(wanted) > 0.95, 'coverage below 95 percent, ids are not matching'

parquet shards:   0%|          | 0/28 [00:00<?, ?it/s]

article coverage: 4479 of 4479 (100.0 percent)


### 7.3 Stage 3, pick the best chunks

Article level retrieval is too coarse to use directly, since a long article is mostly irrelevant
to any one question. The chunks from all candidate articles are pooled and ranked by cosine
similarity against the question and its options together, and the top five become the context.

The cross encoder reranking stage from Milestone 3 is left out. Its weights download from the
Hugging Face hub, which is not reliable in this environment. Milestone 3 measured a 73 percent
top five hit rate for the bi encoder alone, so retrieval remains useful without it, if less
precise. This is a known limitation rather than a design preference.

In [23]:
if BUILD:
    contexts = load_ckpt('contexts_final')
    if contexts is None:
        contexts = []
        for qi in tqdm(range(len(all_rows)), desc='retrieve'):
            row = all_rows[qi]
            seen, chunks = set(), []
            for art_row in article_rows[qi]:
                if art_row in seen:
                    continue
                seen.add(art_row)
                aid = str(wiki_map.iloc[art_row][id_col])
                chunks.extend(article_chunks.get(aid, []))
            if not chunks:
                contexts.append('')
                continue
            q_text = str(row['prompt']) + ' ' + ' '.join(str(row[o]) for o in OPTIONS)
            c_emb = bi_encoder.encode(chunks, batch_size=128, show_progress_bar=False)
            qv = bi_encoder.encode([q_text], show_progress_bar=False)[0]
            sims = c_emb @ qv / (np.linalg.norm(c_emb, axis=1) * np.linalg.norm(qv) + 1e-9)
            top = np.argsort(-sims)[:K_FINAL]
            contexts.append(' '.join(chunks[i] for i in top)[:CTX_CHARS])
        save_ckpt('contexts_final', contexts)

    assert len(contexts) == N_VAL + len(test), 'context count does not match the row count'
    print('contexts ready:', len(contexts), '| empty:', sum(1 for c in contexts if not c))
    print()
    print('example context for validation row 0')
    print(contexts[0][:400], '...')

retrieve:   0%|          | 0/844 [00:00<?, ?it/s]

contexts ready: 844 | empty: 0

example context for validation row 0
Heavy ion fusion. Heavy ion fusion is a fusion energy concept that uses a stream of high-energy ions from a particle accelerator to rapidly heat and compress a small pellet of fusion fuel. It is a subclass of the larger inertial confinement fusion (ICF) approach, replacing the more typical laser systems with an accelerator. Accelerators have the potential to be much more efficient in terms of deli ...


Contexts are cached under `contexts_final.pkl`. The file is tied to the current split, since
positions zero to `N_VAL` are validation rows in their split order. Changing the split or the
seed means deleting the cache and regenerating it, otherwise contexts line up with the wrong
questions.

In [24]:
if BUILD:
    del bi_encoder, wiki_index, wiki_map, q_emb, article_chunks
    gc.collect()
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f'GPU {d}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB')

GPU 0: 15.0 GB free of 15.6 GB
GPU 1: 15.5 GB free of 15.6 GB


## 8. Model 2 and 3: Llama 3.1 8B, retrieval and the ensemble

The base model is Llama 3.1 8B Instruct, loaded in 4 bit NF4 with float16 compute and sharded
across both T4s. Float16 is used rather than bfloat16 because the T4 has no native bfloat16
support. Quantisation is what makes an 8B model fit at all here.

The Milestone 4 QLoRA adapter is pulled through kagglehub. When it loads, four signals are
available. When it does not, the notebook continues with the two base signals rather than
stopping.

In [25]:
if BUILD:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") 

    LLAMA_DIR = "meta-llama/Llama-3.1-8B-instruct"
    ADAPTER_HANDLE = 'dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter'

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True)

    llama_tok = AutoTokenizer.from_pretrained(LLAMA_DIR)
    if llama_tok.pad_token is None:
        llama_tok.pad_token = llama_tok.eos_token

    llama = AutoModelForCausalLM.from_pretrained(
        LLAMA_DIR, quantization_config=bnb_config, device_map='auto', low_cpu_mem_usage=True)
    llama.eval()
    print('base model loaded from', LLAMA_DIR)

    HAS_ADAPTER = False
    try:
        import kagglehub
        from peft import PeftModel
        adapter_dir = kagglehub.model_download(ADAPTER_HANDLE)
        llama = PeftModel.from_pretrained(llama, adapter_dir)
        llama.eval()
        HAS_ADAPTER = True
        print('QLoRA adapter loaded, four signals available')
    except Exception as e:
        print('adapter not loaded:', e)
        print('continuing with two base signals')

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

base model loaded from meta-llama/Llama-3.1-8B-instruct
QLoRA adapter loaded, four signals available


### 8.1 Reading answers from logits

The model is never asked to generate text. The prompt ends at `Answer:` and the logits for the
next token are read directly. Only the five entries corresponding to A to E matter, so those are
taken and softmaxed into a distribution over the options. One forward pass per signal, no
sampling, and the full ranking needed for MAP@3 comes out of it.

Each letter has two token ids, one bare and one with a leading space. The larger logit of the two
is used.

In [26]:
if BUILD:
    letter_ids = {}
    for o in OPTIONS:
        ids = {llama_tok.encode(o, add_special_tokens=False)[-1],
               llama_tok.encode(' ' + o, add_special_tokens=False)[-1]}
        letter_ids[o] = list(ids)
    print('letter token ids:', letter_ids)

    def build_prompt(row, context):
        """Build the prompt, with the retrieved context when one is supplied."""
        q = str(row['prompt'])
        opts = '\n'.join(f'{o}. {row[o]}' for o in OPTIONS)
        if context:
            return f'Context:\n{context}\n\nQuestion: {q}\n\nOptions:\n{opts}\n\nAnswer:'
        return f'Question: {q}\n\nOptions:\n{opts}\n\nAnswer:'

    @torch.no_grad()
    def letter_probs(row, context, use_adapter):
        """Return the probability of each option letter for one question."""
        input_ids = llama_tok(build_prompt(row, context),
                              return_tensors='pt').input_ids.to(llama.device)
        if HAS_ADAPTER and not use_adapter:
            with llama.disable_adapter():
                logits = llama(input_ids).logits[0, -1]
        else:
            logits = llama(input_ids).logits[0, -1]
        raw = torch.tensor([max(logits[i].item() for i in letter_ids[o]) for o in OPTIONS])
        return torch.softmax(raw, dim=0).numpy()

    print('demo, validation row 0 ->',
          top3(letter_probs(val_rows[0], contexts[0], False)),
          '| true answer', val_rows[0]['answer'])

letter token ids: {'A': [32, 362], 'B': [33, 426], 'C': [34, 356], 'D': [35, 423], 'E': [36, 469]}
demo, validation row 0 -> ['A', 'D', 'C'] | true answer A


### 8.2 The four signals

Two variables give four combinations: with or without retrieved context, and base or fine tuned
weights.

| Signal | Context | Weights |
|---|---|---|
| base_rag | retrieved | base |
| base_closed | none | base |
| tuned_rag | retrieved | QLoRA adapter |
| tuned_closed | none | QLoRA adapter |

Ensembling works when the members make different mistakes. Here the differences are real: the
closed book signals rely on what the model already knows, the RAG signals rely on the retrieved
passage, and the tuned signals have seen the answer format during fine tuning. Averaging the four
probability arrays lets a confident signal outvote an uncertain one.

In [27]:
if BUILD:
    SIGNALS = ['base_rag', 'base_closed'] + (['tuned_rag', 'tuned_closed'] if HAS_ADAPTER else [])

    def all_signal_probs(row, context):
        """Return a dict of probability arrays, one per available signal."""
        out = {'base_rag': letter_probs(row, context, False),
               'base_closed': letter_probs(row, '', False)}
        if HAS_ADAPTER:
            out['tuned_rag'] = letter_probs(row, context, True)
            out['tuned_closed'] = letter_probs(row, '', True)
        return out

    val_sig = {s: [] for s in SIGNALS}
    for i in tqdm(range(N_VAL), desc='validate'):
        p = all_signal_probs(val_rows[i], contexts[i])
        for s in SIGNALS:
            val_sig[s].append(p[s])

    val_true = [r['answer'] for r in val_rows]
    val_scores = {}
    for s in SIGNALS:
        val_scores[s] = map_at_3([top3(p) for p in val_sig[s]], val_true)

    val_ens = np.mean([np.array(val_sig[s]) for s in SIGNALS], axis=0)
    val_scores['ensemble'] = map_at_3([top3(p) for p in val_ens], val_true)

    for name, score in val_scores.items():
        print(f'{name:14s} validation MAP@3: {score:.4f}')

    np.save(f'{WORK}/val_probs_ensemble.npy', val_ens)
    save_ckpt('val_scores', val_scores)

validate:   0%|          | 0/344 [00:00<?, ?it/s]

base_rag       validation MAP@3: 0.8706
base_closed    validation MAP@3: 0.8149
tuned_rag      validation MAP@3: 1.0000
tuned_closed   validation MAP@3: 0.9869
ensemble       validation MAP@3: 0.9898


### 8.3 Test set probabilities

Every signal is computed once on the test set and saved. Submissions then read these arrays
rather than rerunning the model, so the ensemble reproduces exactly and a submission needs no
GPU.

In [28]:
if BUILD:
    test_sig = {s: [] for s in SIGNALS}
    for i in tqdm(range(len(test)), desc='test'):
        p = all_signal_probs(test.iloc[i], contexts[N_VAL + i])
        for s in SIGNALS:
            test_sig[s].append(p[s])

    for s in SIGNALS:
        arr = np.array(test_sig[s])
        np.save(f'{WORK}/test_probs_{s}.npy', arr)
        print('saved', f'test_probs_{s}.npy', arr.shape)

    test_ens = np.mean([np.array(test_sig[s]) for s in SIGNALS], axis=0)
    np.save(f'{WORK}/test_probs_ensemble.npy', test_ens)
    print('saved test_probs_ensemble.npy', test_ens.shape)

test:   0%|          | 0/500 [00:00<?, ?it/s]

saved test_probs_base_rag.npy (500, 5)
saved test_probs_base_closed.npy (500, 5)
saved test_probs_tuned_rag.npy (500, 5)
saved test_probs_tuned_closed.npy (500, 5)
saved test_probs_ensemble.npy (500, 5)


In [29]:
if BUILD:
    import wandb
    run = wandb.init(entity='22f3000114-dl-genai-project', project='dl-genai-project',
                     name='llama-ensemble-final', group='final-notebook',
                     config={'base': 'llama-3.1-8b-instruct 4bit', 'adapter': HAS_ADAPTER,
                             'retrieval': 'MiniLM bi encoder, no rerank',
                             'k_articles': K_ARTICLES, 'k_final': K_FINAL,
                             'split': 'GroupShuffleSplit on core_q'}, reinit=True)
    run.log({f'val_map3_{k}': v for k, v in val_scores.items()})
    run.finish()
    print('logged to W&B')

wandb: setting up run vk1pofpw
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260721_165214-vk1pofpw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run llama-ensemble-final
wandb: ⭐️ View project at https://wandb.ai/22f3000114-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/22f3000114-dl-genai-project/dl-genai-project/runs/vk1pofpw
wandb: updating run metadata; uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb:  val_map3_base_closed ▁
wandb:     val_map3_base_rag ▁
wandb:     val_map3_ensemble ▁
wandb: val_map3_tuned_closed ▁
wandb:    val_map3_tuned_rag ▁
wandb: 
wandb: Run summary:
wandb:  val_map3_base_closed 0.81492
wandb:     val_map3_base_rag 0.87064
wandb:     val_map3_ensemble 0.98983
wandb: val_map3_tuned_closed 0.98692
wandb:    val_map3_tuned_rag 1
wandb: 
wandb: 🚀 View run llama-ensemble-final at: https://wandb.ai/22f3000114-dl

logged to W&B


## 9. Model comparison

All three models are scored on the same validation set with the same metric, which is what makes
the comparison meaningful. The from scratch model is the reference point for what the task looks
like without any pretrained knowledge.

In [30]:
if BUILD:
    rows = [{'model': 'BiLSTM from scratch', 'type': 'scratch', 'val_map3': best_map3}]
    for s in SIGNALS:
        rows.append({'model': f'Llama 3.1 8B, {s}', 'type': 'pretrained', 'val_map3': val_scores[s]})
    rows.append({'model': 'Four signal ensemble', 'type': 'ensemble', 'val_map3': val_scores['ensemble']})

    comparison = pd.DataFrame(rows).sort_values('val_map3', ascending=False).reset_index(drop=True)
    comparison.to_csv(f'{WORK}/model_comparison.csv', index=False)
    print(comparison.round(4).to_string(index=False))

                     model       type  val_map3
       BiLSTM from scratch    scratch    1.0000
   Llama 3.1 8B, tuned_rag pretrained    1.0000
      Four signal ensemble   ensemble    0.9898
Llama 3.1 8B, tuned_closed pretrained    0.9869
    Llama 3.1 8B, base_rag pretrained    0.8706
 Llama 3.1 8B, base_closed pretrained    0.8149


Fill in after the build run. Points worth stating in the report:

- where the from scratch model lands against the majority class baseline from section 3
- what retrieval adds, by comparing base_rag against base_closed
- what fine tuning adds, by comparing the tuned signals against the base ones
- whether the ensemble beats its best member, and by how much

## 10. Submission

Submissions read the saved arrays. Set `SUBMIT_MODEL` at the top, run with `BUILD = False`, and
this section writes `submission.csv`. Because the arrays are fixed on disk, the same selection
always produces the same file.

Three submissions are planned:

1. `bilstm`, the from scratch model, as the floor
2. `base_rag`, pretrained with retrieval, no fine tuning
3. `ensemble`, all four signals averaged

In [31]:
SUBMIT_FILES = {
    'bilstm':   'test_probs_bilstm.npy',
    'base_rag': 'test_probs_base_rag.npy',
    'ensemble': 'test_probs_ensemble.npy',
}

def write_submission(model_key):
    """Load the saved probabilities for a model and write submission.csv."""
    path = f'{WORK}/{SUBMIT_FILES[model_key]}'
    if not os.path.exists(path):
        hits = glob.glob(f'/kaggle/input/**/{SUBMIT_FILES[model_key]}', recursive=True)
        if not hits:
            raise FileNotFoundError(f'{path} not found, run the notebook with BUILD = True first')
        path = hits[0]

    probs = np.load(path)
    assert len(probs) == len(test), f'expected {len(test)} rows, found {len(probs)}'

    submission = pd.DataFrame({
        'id': test['id'],
        'prediction': [' '.join(top3(p)) for p in probs]})
    submission.to_csv('submission.csv', index=False)
    print('model     :', model_key)
    print('source    :', path)
    print('rows      :', len(submission))
    print(submission.head())
    return submission

submission = write_submission(SUBMIT_MODEL)

model     : ensemble
source    : /kaggle/working/test_probs_ensemble.npy
rows      : 500
   id prediction
0   1      D A E
1   2      B A D
2   3      B C A
3   4      E A C
4   5      C A B


## 11. Notes

### What is in this notebook

Three models of genuinely different architectures. The BiLSTM is written and trained here with no
pretrained weights. Llama 3.1 8B is a pretrained decoder used through logit extraction rather
than generation. The ensemble averages four signals that differ in both context and weights.

### The data problem and how it was handled

The training file has 2000 rows but around 1125 distinct questions, because a smaller pool was
wrapped in several instruction phrases. Splitting rows independently puts the same question on
both sides and inflates every validation score. Section 4 groups on the stripped question text and
asserts zero overlap. This was found by noticing a perfect validation score that no honest model
should reach on this task.

### Known limitations

Cross encoder reranking is missing because its weights cannot be downloaded reliably here, so
retrieval is bi encoder only at roughly a 73 percent top five hit rate. Contexts are capped at
4500 characters. The validation set is small enough that differences of one or two points are
within noise.

### Milestone mapping

Milestone 1 is the EDA and baselines in section 3. Milestone 2 is the similarity baselines that
motivated retrieval. Milestone 3 is the retrieval pipeline in section 7. Milestone 4 is the QLoRA
adapter behind the tuned signals. Milestone 5 is the ensembling and MAP@3 work in section 8.